In [1]:
from cube_datasets import TrainingValueDataset
from cube_nn import CubeValueResNetV2
from training import train_on_value_dataset
from torch import optim
import matplotlib.pyplot as plt
import torch
from cube_nn import NNValueFunctionType

# device = "cpu"
device = "cuda"

In [2]:
N_MOVES_MAX = 20
N_ROLLOUTS = 20000

# Create a new netcreate_balanced
net = CubeValueResNetV2()
net = net.to(device)
initial_dataset = TrainingValueDataset.create_from_trajectories(net.get_cube_to_tensor(), N_ROLLOUTS, N_MOVES_MAX, device=device)
optimizer = optim.Adam(net.parameters(), lr=0.001)
losses = []

initial_loss = train_on_value_dataset(net, initial_dataset, optimizer, n_epochs=10, batch_size=200, device=device)
losses.append(initial_loss)
print(f"Initial loss: {initial_loss}")

i = 0

Epoch 1/10, Loss: 10.182955842494964


Epoch 2/10, Loss: 7.128581622123718


Epoch 3/10, Loss: 6.532971673488617


Epoch 4/10, Loss: 5.941244571208954


Epoch 5/10, Loss: 5.250423708081246


Epoch 6/10, Loss: 4.418603914737702


Epoch 7/10, Loss: 3.557217008113861


Epoch 8/10, Loss: 2.841615169048309


Epoch 9/10, Loss: 2.322101207137108


Epoch 10/10, Loss: 2.0000616360902788
Initial loss: 143.10746644592285


In [3]:
# Load a previously saved net
i = 200
net = CubeValueResNetV2()
net.load_state_dict(torch.load(f'temp_models/resnetV2/cube_value_resnetV2_iter_{i}.pth'))
net = net.to(device)

optimizer = optim.Adam(net.parameters(), lr=0.001)

losses = torch.load(f'temp_models/resnetV2/losses_list_{i}.pth')

/tmp/ipykernel_1230710/950408107.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load(f'temp_models/resnetV2/cube_value_resnetV2_iter_{i}.pth')

In [4]:
N_CUBES_PER_MOVE = 1000
N_ROLLOUTS = 20000
N_MOVES_MAX = 20
N_MOVES_DIFFERENCE_MAX = 3
N_EPOCHS = 2
BATCH_SIZE = 100
SIMILAR_EXTENSION_FRACTION = None

# set the STANDARD value function for speed
net.set_value_function_type(NNValueFunctionType.STANDARD)
net.to(device)

for j in range(100):
    i += 1
    # new_dataset = TrainingValueDataset.create_from_bellman_equation(net, N_MOVES_MAX, N_CUBES_PER_MOVE, n_moves_difference_max=N_MOVES_DIFFERENCE_MAX, similar_cubes_extension=SIMILAR_EXTENSION_FRACTION)
    # new_dataset = TrainingValueDataset.create_balanced(net.get_cube_to_tensor(), N_MOVES_MAX, N_CUBES_PER_MOVE)
    new_dataset = TrainingValueDataset.create_from_trajectories(net.get_cube_to_tensor(), N_ROLLOUTS, N_MOVES_MAX, device=device)
    loss = train_on_value_dataset(net, new_dataset, optimizer, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, device=device)
    losses.append(loss)
    print(f"Iteration {i+1}, loss: {loss}")

    # Save the model and plot of losses every 10 iterations
    if (i + 1) % 10 == 0:
        torch.save(net.state_dict(), f'temp_models/resnetV2/cube_value_resnetV2_iter_{i+1}.pth')
        torch.save(losses, f'temp_models/resnetV2/losses_list_{i+1}.pth')
        plt.plot(losses[1:]) # Skip initial loss for better visualization
        plt.xlabel('Iteration')
        plt.ylabel('Loss')
        plt.title('Training Loss Over Iterations')
        plt.savefig(f'figs/resnetV2/loss_plot_iter_{i+1}.png')
        plt.clf()

Epoch 1/2, Loss: 4.8823621609210965


Epoch 2/2, Loss: 4.233477119833231
Iteration 202, loss: 4.943369650870562


Epoch 1/2, Loss: 4.908740420162678


Epoch 2/2, Loss: 4.242448521614075
Iteration 203, loss: 5.042469676405191


Epoch 1/2, Loss: 4.958661077260971


...

Epoch 2/2, Loss: 4.135470835447311
Iteration 299, loss: 4.913285052239895


Epoch 1/2, Loss: 4.76409137904644


Epoch 2/2, Loss: 4.129838530063629
Iteration 300, loss: 4.9004788926243785


Epoch 1/2, Loss: 4.80454921078682


Epoch 2/2, Loss: 4.159961535394192
Iteration 301, loss: 4.961087349802256


In [3]:
from cube import Cube
from solvers import AStarSolver
n = 8
random_cube = Cube(n_scramble_moves=n)

value_function = net.as_value_function()

print(f"value of random cube ({n} moves):", value_function(random_cube))

solver = AStarSolver(value_function=value_function, weight=0.2, max_queue_size=1000000, max_moves=50, t_max=60.0)

solution = solver(random_cube)
print("solution moves:", solution)

value of random cube (8 moves): 6.163855
solution moves: None
